# Practical 3: Clean the Data and Preprocess for Further Analysis

**Objective:** Clean structured data for analysis.

- Convert timestamp to datetime format
- Handle missing values and inconsistent entries
- Normalize URL paths (e.g., `/index.html` vs `/index`)
- Lowercase strings and remove extraneous spaces

In [1]:
import pandas as pd

df = pd.read_csv('logs/structured_access_log.csv')
print('Rows loaded:', len(df))
df.head()

Rows loaded: 47685


,IP Address,Date/Time,Request Type,Resource,Protocol,Status Code,Bytes Sent,Referrer,User Agent
0,144.187.77.221,01/Aug/2026:00:00:25 +0530,GET,/api/products,HTTP/1.1,404,418,https://facebook.com,Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like M...
1,94.87.216.251,01/Aug/2026:00:02:59 +0530,GET,/index.html,HTTP/1.1,304,8710,https://bing.com,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
2,192.168.1.168,01/Aug/2026:00:03:15 +0530,GET,/about,HTTP/1.1,200,463,-,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
3,72.23.185.21,01/Aug/2026:00:03:16 +0530,GET,/images/logo.png,HTTP/1.1,200,2243,https://twitter.com,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
4,176.253.49.242,01/Aug/2026:00:04:04 +0530,POST,/style.css,HTTP/1.1,200,1414,https://example.com,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...


## 1. Convert timestamp to datetime format

In [2]:
# Format looks like: 01/Aug/2026:00:02:59 +0530
df['Date/Time'] = pd.to_datetime(df['Date/Time'], format='%d/%b/%Y:%H:%M:%S %z')
print(df['Date/Time'].dtype)
df['Date/Time'].head()

datetime64[ns, UTC+05:30]


0   2026-08-01 00:00:25+05:30
1   2026-08-01 00:02:59+05:30
2   2026-08-01 00:03:15+05:30
3   2026-08-01 00:03:16+05:30
4   2026-08-01 00:04:04+05:30
Name: Date/Time, dtype: datetime64[ns, UTC+05:30]

## 2. Handle missing values and inconsistent entries

In [3]:
print('Missing values per column:')
print(df.isnull().sum())

# Referrer field legitimately uses '-' to mean 'no referrer' (direct visit),
# not a missing value in the pandas sense — normalize it to an explicit label
# instead of leaving it as a literal '-' string.
df['Referrer'] = df['Referrer'].replace('-', 'direct')

# Drop rows that are missing any of the truly essential fields
essential = ['IP Address', 'Date/Time', 'Request Type', 'Resource', 'Status Code']
before = len(df)
df = df.dropna(subset=essential)
print(f'Dropped {before - len(df)} rows with missing essential fields')

Missing values per column:
IP Address      0
Date/Time       0
Request Type    0
Resource        0
Protocol        0
Status Code     0
Bytes Sent      0
Referrer        0
User Agent      0
dtype: int64
Dropped 0 rows with missing essential fields


## 3. Normalize URL paths

URL-decode percent-encoded characters, strip query strings into a separate check, and treat trailing-slash / extension variants consistently so `/index.html`, `/index`, and `/index/` aren't treated as different resources downstream.

In [4]:
from urllib.parse import unquote

def normalize_path(resource: str) -> str:
    path = unquote(resource)          # decode %2e%2e%2f -> ../ etc.
    path = path.strip()
    if '?' in path:
        path = path.split('?', 1)[0]  # drop query string for the normalized version
    if len(path) > 1 and path.endswith('/'):
        path = path.rstrip('/')       # /about/ -> /about
    return path

df['Resource Normalized'] = df['Resource'].apply(normalize_path)
df[['Resource', 'Resource Normalized']].drop_duplicates().head(10)

,Resource,Resource Normalized
0,/api/products,/api/products
1,/index.html,/index.html
2,/about,/about
3,/images/logo.png,/images/logo.png
4,/style.css,/style.css
5,/search?q=laptop,/search
6,/help,/help
7,/register,/register
8,/blog/post-1,/blog/post-1
9,/products/1,/products/1


## 4. Lowercase strings and remove extraneous spaces

In [5]:
text_cols = ['Request Type', 'Referrer', 'User Agent']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Request Type (GET/POST) is conventionally uppercase in HTTP — keep it that way
df['Request Type'] = df['Request Type'].str.upper()

# Referrer and User Agent are safe to lowercase for consistent matching later
df['Referrer'] = df['Referrer'].str.lower()
df['User Agent'] = df['User Agent'].str.lower()

df.head()

,IP Address,Date/Time,Request Type,Resource,Protocol,Status Code,Bytes Sent,Referrer,User Agent,Resource Normalized
0,144.187.77.221,2026-08-01 00:00:25+05:30,GET,/api/products,HTTP/1.1,404,418,https://facebook.com,mozilla/5.0 (iphone; cpu iphone os 17_0 like m...,/api/products
1,94.87.216.251,2026-08-01 00:02:59+05:30,GET,/index.html,HTTP/1.1,304,8710,https://bing.com,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/index.html
2,192.168.1.168,2026-08-01 00:03:15+05:30,GET,/about,HTTP/1.1,200,463,direct,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/about
3,72.23.185.21,2026-08-01 00:03:16+05:30,GET,/images/logo.png,HTTP/1.1,200,2243,https://twitter.com,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/images/logo.png
4,176.253.49.242,2026-08-01 00:04:04+05:30,POST,/style.css,HTTP/1.1,200,1414,https://example.com,mozilla/5.0 (windows nt 10.0; win64; x64) appl...,/style.css


## Final check

In [6]:
print('Rows remaining:', len(df))
print('\nMissing values per column:')
print(df.isnull().sum())
df.info()

Rows remaining: 47685

Missing values per column:
IP Address             0
Date/Time              0
Request Type           0
Resource               0
Protocol               0
Status Code            0
Bytes Sent             0
Referrer               0
User Agent             0
Resource Normalized    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47685 entries, 0 to 47684
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype                    
---  ------               --------------  -----                    
 0   IP Address           47685 non-null  object                   
 1   Date/Time            47685 non-null  datetime64[ns, UTC+05:30]
 2   Request Type         47685 non-null  object                   
 3   Resource             47685 non-null  object                   
 4   Protocol             47685 non-null  object                   
 5   Status Code          47685 non-null  int64                    
 6   Bytes Sent           47685 no

## Save cleaned dataset for Practical 4

In [7]:
df.to_csv('logs/cleaned_access_log.csv', index=False)
print('Saved logs/cleaned_access_log.csv')

Saved logs/cleaned_access_log.csv
